# Defining methods

In [13]:
import numpy as np
import scipy as sc
import scipy.sparse as sp
from petsc4py import PETSc
from slepc4py import SLEPc

def create_scipy_sparse_matrix(data):
    row = data[:, 0].astype(int)
    col = data[:, 1].astype(int)
    val = data[:, 2]
    size = int(max(row.max(), col.max()) + 1)
    return sp.csr_matrix((val, (row, col)), shape=(size, size))

# Function to create PETSc sparse matrix from data
def create_petsc_matrix(data):
    # Extract rows, columns, and values
    rows = data[:, 0].astype(int) - 1  # Convert to zero-based index
    cols = data[:, 1].astype(int) - 1  # Convert to zero-based index
    values = data[:, 2]

    # Create PETSc matrix
    nrows = rows.max() + 1
    ncols = cols.max() + 1
    A = PETSc.Mat().createAIJ([nrows, ncols], nnz=3)
    A.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    A.setUp()
    for r, c, v in zip(rows, cols, values):
        A[r, c] = v
    A.assemble()
    return A

# Function to convert a SciPy sparse matrix to a PETSc matrix
def scipy_to_petsc(scipy_matrix):
    # coo = scipy_matrix.tocoo()
    # petsc_matrix = PETSc.Mat().createAIJ(size=scipy_matrix.shape, csr=(coo.row.astype(np.int32), coo.col.astype(np.int32), coo.data))
    # petsc_matrix.setValuesCSR(coo.row, coo.col, coo.data)
    petsc_mat = PETSc.Mat().createAIJ(size=scipy_matrix.shape, csr=(scipy_matrix.indptr, scipy_matrix.indices, scipy_matrix.data))
    petsc_mat.assemble()
    return petsc_mat

def compute_invdiag(A):
    diagA = A.getDiagonal()
    diagA_inv = diagA.copy()

    # Compute the lumped row reciprocal of each diagonal entry manually
    for i in range(diagA_inv.size):
        rowsum = 0.0
        for j in range(diagA_inv.size):
            rowsum += A[i, j]
        if rowsum != 0:
            diagA_inv[i] = 1.0 / rowsum
        else:
            print("Row sum is zero for row", i)
            break

    # Create a diagonal matrix D_inv from the inverse diagonal values
    D_inv = PETSc.Mat().createAIJ(A.getSize(), comm=comm)
    D_inv.setUp()
    for i in range(diagA_inv.size):
        D_inv.setValue(i, i, diagA_inv[i])
    D_inv.assemble()
    
    return D_inv

In [14]:
from mpi4py import MPI
import sys

comm = MPI.COMM_WORLD

def par_print(string):
    if comm.rank == 0:
        print(string)
        sys.stdout.flush()

def eigenvalues(A, B, shift=0.0, n_eigs=11, precond = False):
    # Check if need precond
    if precond == True:
        D_inv = compute_invdiag(A+B)
        # D_inv = compute_invdiag(A)
        A = D_inv.matMult(A)
        B = D_inv.matMult(B)


    # Create SLEPc Eigenvalue solver
    eps = SLEPc.EPS().create(comm)
    eps.setOperators(A, B)
    eps.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)
    eps.setWhichEigenpairs(eps.Which.TARGET_MAGNITUDE)
    eps.setTarget(shift)

    st = eps.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(shift)

    eps.setDimensions(n_eigs, PETSc.DECIDE, PETSc.DECIDE)
    eps.setFromOptions()
    eps.solve()

    its = eps.getIterationNumber()
    eps_type = eps.getType()
    n_ev, n_cv, mpd = eps.getDimensions()
    tol, max_it = eps.getTolerances()
    n_conv = eps.getConverged()

    par_print(f"Number of iterations: {its}")
    par_print(f"Solution method: {eps_type}")
    par_print(f"Number of requested eigenvalues: {n_ev}")
    par_print(f"Stopping condition: tol={tol}, maxit={max_it}")
    par_print(f"Number of converged eigenpairs: {n_conv}")

    computed_eigenvalues = []
    for i in range(min(n_conv, n_eigs)):
        lmbda = eps.getEigenvalue(i)
        # computed_eigenvalues.append(np.round(np.real(lmbda), 1))
        computed_eigenvalues.append(np.round(np.real(lmbda), 5))

    eps.destroy()

    # Print results
    # np.set_printoptions(formatter={'float': '{:5.1f}'.format})
    eigenvalues_exact = np.array([2.0, 2.0, 2.0, 3.0, 3.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0])
    computed_eigenvalues = np.sort(computed_eigenvalues)
    
    par_print(f"Exact    = {eigenvalues_exact}")
    par_print(f"Nédélec  = {computed_eigenvalues}")
    return computed_eigenvalues

In [15]:
def preprocess_A_3field(A_data, num_edges, num_faces, num_cells):
    A = create_scipy_sparse_matrix(A_data)

    # Extract submatrices W, M, and B from A
    M = -A[1:num_edges, 1:num_edges]  # edges x edges
    W = A[num_edges:num_edges+num_faces, 1:num_edges]  # faces x edges
    B = A[num_edges + num_faces:num_edges + num_faces + num_cells, num_edges:num_edges+num_faces]  # cells x faces
    # print(M.shape, W.shape, B.shape)

    # Compute W * M^{-1}
    WT = W.transpose()  # edges x faces
    M_invWT = sc.sparse.linalg.spsolve(M, WT)  # edges x faces

    # Compute W * M^{-1} * W^T
    WMWT = W @ M_invWT

    # Transpose B to get B^T
    BT = B.transpose()

    # Determine the sizes for the resulting block matrix
    m, n = WMWT.shape  # Assume WMWT is mxm, m=num_faces
    p, q = B.shape  # Assume B is pxm, p=num_cells

    # Create the resulting block matrix
    result = sp.lil_matrix((m+p, m+p))

    # Insert submatrices into the block matrix
    # Insert W * M^{-1} * W^T into the top-left block
    result[:m, :m] = WMWT

    # Insert B^T into the top-right block
    result[:m, m:m + p] = BT

    # Insert B into the bottom-left block
    result[m:m + p, :m] = B

    # Convert the resulting block matrix to PETSc
    petsc_result = scipy_to_petsc(result.tocsr())

    return petsc_result

def preprocess_B_3field(B_data, num_edges, num_faces, num_cells):
    B = create_petsc_matrix(B_data)

    # Define row and column indices for the submatrices
    row_indices_M = list(range(num_edges, num_edges+num_faces+num_cells))
    col_indices_M = list(range(num_edges, num_edges+num_faces+num_cells))

    # Convert indices to PETSc IS (Index Set)
    row_IS_M = PETSc.IS().createGeneral(row_indices_M)
    col_IS_M = PETSc.IS().createGeneral(col_indices_M)

    # Extract submatrices M from A
    # M = B.createSubMatrix(row_IS_M, col_IS_M)

    # Create the resulting block matrix
    block_size = num_faces + num_cells
    result = PETSc.Mat().createAIJ([block_size, block_size])
    result.setUp()

    # Insert submatrices into the block matrix
    for i in range(block_size):
        for j in range(block_size):
            result[i, j] = B[num_edges+i, num_edges+j]

    # Finalize the assembly of the block matrix
    result.assemble()
    return result

# Coarsest mesh eigenvalues

In [35]:
# Read the matrix data
# i = 0
A_data = np.loadtxt("../../build/A0.dat")
B_data = np.loadtxt("../../build/B0.dat")

# Create the PETSc matrices A and B
A = create_petsc_matrix(A_data)
B = create_petsc_matrix(B_data)

# OPTIONAL FOR 3FIELD
# A = preprocess_A_3field(A_data, 1854, 2808, 1296)
# B = preprocess_B_3field(B_data, 1854, 2808, 1296)

# Compute evals
evals = eigenvalues(A, B, 0.5, 343+200) # dim Lagr = 343
# evals = eigenvalues(A, B, 3.2, 41)
# evals = eigenvalues(A, B, 1.1, 11, True)


# evals = evals[..., None]
# np.savetxt(sys.stdout, evals, fmt="%.1f")

Number of iterations: 3
Solution method: krylovschur
Number of requested eigenvalues: 543
Stopping condition: tol=1e-08, maxit=100
Number of converged eigenpairs: 676
Exact    = [2. 2. 2. 3. 3. 5. 5. 5. 5. 5. 5.]
Nédélec  = [-4.300000e-04 -2.800000e-04 -2.700000e-04 -2.000000e-04 -1.800000e-04
 -1.400000e-04 -1.300000e-04 -1.300000e-04 -1.200000e-04 -1.100000e-04
 -1.000000e-04 -1.000000e-04 -8.000000e-05 -8.000000e-05 -7.000000e-05
 -6.000000e-05 -6.000000e-05 -4.000000e-05 -4.000000e-05 -4.000000e-05
 -4.000000e-05 -3.000000e-05 -3.000000e-05 -3.000000e-05 -3.000000e-05
 -3.000000e-05 -3.000000e-05 -2.000000e-05 -2.000000e-05 -2.000000e-05
 -2.000000e-05 -2.000000e-05 -1.000000e-05 -1.000000e-05 -1.000000e-05
 -1.000000e-05 -1.000000e-05 -1.000000e-05 -1.000000e-05  0.000000e+00
  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
  0.000000e+00  0.000000e+00 -0.000000e+00 -0.000000e+00 -0.000000e+00
 -0.000000e+00 -0.000000e+00 -0.000000e+00 -0.000000e+00 -0.000000

In [ ]:
# Compute evals
evals = eigenvalues(A, B, 3.5, 343+200) # dim Lagr = 343


# Finest mesh eigenvalues

In [45]:
# i = 1
A_data = np.loadtxt("../../build/A1.dat")
B_data = np.loadtxt("../../build/B1.dat")

# Create the PETSc matrices A and B
A = create_petsc_matrix(A_data)
B = create_petsc_matrix(B_data)

evals = eigenvalues(A, B, 0.5, 1728+20) # dim Lagr = 2197
# evals = eigenvalues(A, B, 3.2, 41)
# evals = eigenvalues(A, B, 1.1, 11, True)


# evals = evals[..., None]
np.savetxt(sys.stdout, evals, fmt="%.1f")

Number of iterations: 30
Solution method: krylovschur
Number of requested eigenvalues: 1748
Stopping condition: tol=1e-08, maxit=100
Number of converged eigenpairs: 1863
Exact    = [2. 2. 2. 3. 3. 5. 5. 5. 5. 5. 5.]
Nédélec  = [-1.50000e-04 -1.30000e-04 -1.10000e-04 ...  7.81501e+00  7.98275e+00
  8.76670e+00]
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
-0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0

# Printing eigenvalue results

In [36]:
# evals_pos = evals[evals[0:-1]>0]
# np.sort(evals_pos)
# [x for x in evals[1:343]]

# [x for x in evals[430:450]]
[x for x in evals[250:350]]
# [x for x in evals[1725:1728+20]]


[0.00016,
 0.00018,
 0.00019,
 0.00021,
 0.00032,
 0.0004,
 0.9585,
 0.99421,
 2.08209,
 2.23567,
 2.44711,
 2.79283,
 3.00254,
 3.11083,
 4.11744,
 4.34988,
 4.72528,
 5.02498,
 5.22411,
 5.37185,
 5.4691,
 5.48246,
 5.67685,
 5.88741,
 6.33974,
 6.45426,
 6.79392,
 6.90491,
 7.20072,
 7.29729,
 8.05825,
 8.18717,
 8.36531,
 8.41085,
 8.71803,
 8.73335,
 8.8838,
 9.05896,
 9.21229,
 9.28367,
 9.521,
 9.62439,
 9.77271,
 9.80023,
 9.88613,
 10.08641,
 10.23472,
 10.41724,
 10.46045,
 10.75291,
 10.90215,
 10.96307,
 11.23587,
 11.74659,
 11.76642,
 11.82602,
 11.92458,
 12.1408,
 12.34283,
 12.45722,
 12.51647,
 12.7551,
 12.80878,
 12.90348,
 12.99072,
 13.08021,
 13.32192,
 13.43908,
 13.45129,
 13.58219,
 13.78863,
 13.80169,
 13.94433,
 14.01098,
 14.11116,
 14.18076,
 14.41827,
 14.49798,
 14.59101,
 14.81764,
 14.88167,
 15.1489,
 15.18614,
 15.41009,
 15.42629,
 15.45643,
 15.55058,
 15.60615,
 15.72109,
 15.84269,
 15.91969,
 16.07484,
 16.16314,
 16.32866,
 16.50322,
 16.57975

# Debugging matrices

In [ ]:
Ad = A.convert('dense').getDenseArray()
Bd = B.convert('dense').getDenseArray()

size = Bd.shape
# Bd[size[0]-2,size[1]-2]
# np.diag(Bd)[2808:size[0]]

# Ad[1968-1, 1968-1]
# Ad.diagonal()[1968-21 : 1968+21]
Ad[1,0:100]